In [ ]:
# Line 1: We install kagglehub, which is our automatic robot delivery truck.
!pip install kagglehub

# Line 4: We bring the robot delivery truck into our script.
import kagglehub

# Line 7: We tell the truck to drive to Kaggle and download the newest 'fruits' dataset.
path = kagglehub.dataset_download("moltean/fruits")

# Line 10: Print out the secret folder address where the truck dropped off the photos.
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fruits' dataset.
Path to dataset files: /kaggle/input/fruits


In [ ]:
# Line 1-3: We bring in toolboxes that let us copy files and search inside folders.
import os
import shutil
import glob

# Line 6-7: We tell the computer where our downloaded files are, and make a brand new folder called 'fruit_images'.
DOWNLOADED_PATH = path
CLEAN_IMAGES_DIR = "fruit_images"
os.makedirs(CLEAN_IMAGES_DIR, exist_ok=True)

# Line 11: These are the exact 5 fruit folders we want to look for.
TARGET_CLASSES = ["Apple", "Banana", "Orange", "Mango", "Granadilla"]

print("--- Extracting a Clean Sample of 150 Images ---")

# Line 17: We want exactly 30 pictures of each fruit type.
IMAGES_PER_CLASS = 30

# Line 20: Start looping through our 5 target fruits one by one.
for fruit in TARGET_CLASSES:
    # Line 22-23: Search inside the huge downloaded folder for subfolders matching our fruit name.
    search_path = os.path.join(DOWNLOADED_PATH, "**", f"{fruit}*")
    matching_folders = [f for f in glob.glob(search_path, recursive=True) if os.path.isdir(f)]

    if not matching_folders:
        print(f"⚠ Could not find a folder for: {fruit}")
        continue

    # Line 30: Grab the very first folder that matches (like 'Apple Braeburn').
    source_folder = matching_folders[0]

    # Line 33: Gather all the image files sitting inside that folder.
    all_images = glob.glob(os.path.join(source_folder, "*.jpg")) + glob.glob(os.path.join(source_folder, "*.png"))

    # Line 36: Slice the list so we only keep the first 30 images.
    sample_images = all_images[:IMAGES_PER_CLASS]

    # Line 39: Loop through our 30 selected images.
    for i, img_path in enumerate(sample_images):
        # Line 41: Give each image a neat new name, like 'apple_001.jpg'.
        new_name = f"{fruit.lower()}_{i:03d}.jpg"
        destination = os.path.join(CLEAN_IMAGES_DIR, new_name)
        # Line 44: Copy the image out of the giant messy pile and into our clean folder.
        shutil.copy(img_path, destination)

    print(f"✓ Copied {len(sample_images)} images from '{os.path.basename(source_folder)}' into {CLEAN_IMAGES_DIR}")

# Line 49: Show the final scoreboard count!
print(f"\n🎉 Done! Total clean images ready for Label Studio: {len(os.listdir(CLEAN_IMAGES_DIR))}")

--- Extracting a Clean Sample of 150 Images ---
✓ Copied 30 images from 'Apple Crimson Snow 1' into fruit_images
✓ Copied 30 images from 'Banana 4' into fruit_images
✓ Copied 30 images from 'Orange 3' into fruit_images
✓ Copied 30 images from 'Mangostan 1' into fruit_images
✓ Copied 30 images from 'Granadilla 1' into fruit_images

🎉 Done! Total clean images ready for Label Studio: 150


In [ ]:
!zip -r fruit_images.zip fruit_images

updating: fruit_images/ (stored 0%)
updating: fruit_images/granadilla_003.jpg (deflated 3%)
updating: fruit_images/apple_027.jpg (deflated 3%)
updating: fruit_images/banana_011.jpg (deflated 3%)
updating: fruit_images/banana_026.jpg (deflated 3%)
updating: fruit_images/apple_009.jpg (deflated 3%)
updating: fruit_images/mango_026.jpg (deflated 4%)
updating: fruit_images/orange_028.jpg (deflated 3%)
updating: fruit_images/mango_007.jpg (deflated 4%)
updating: fruit_images/banana_028.jpg (deflated 3%)
updating: fruit_images/mango_023.jpg (deflated 4%)
updating: fruit_images/apple_003.jpg (deflated 4%)
updating: fruit_images/orange_015.jpg (deflated 3%)
updating: fruit_images/granadilla_027.jpg (deflated 5%)
updating: fruit_images/mango_014.jpg (deflated 3%)
updating: fruit_images/orange_026.jpg (deflated 3%)
updating: fruit_images/orange_008.jpg (deflated 3%)
updating: fruit_images/orange_019.jpg (deflated 3%)
updating: fruit_images/banana_012.jpg (deflated 4%)
updating: fruit_images/mang

In [2]:
# Line 1-3: Load the libraries needed to read text files and list items.
import os
import glob
from collections import Counter

# Line 6-8: Setup paths pointing to our newly uploaded final images and text labels.
DATASET_DIR = "final_dataset"
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
LABELS_DIR = os.path.join(DATASET_DIR, "labels")

# Line 11: The standard index numbers YOLO uses for our labels.
# Note: 'Granadilla' from Kaggle is labeled as 'Grapes' in Label Studio!
CLASS_MAP = {0: "Apple", 1: "Banana", 2: "Orange", 3: "Mango", 4: "Grapes"}

def validate_dataset():
    # Line 15-16: Find every single image file and every single label text file.
    image_files = glob.glob(os.path.join(IMAGES_DIR, "*.*"))
    label_files = glob.glob(os.path.join(LABELS_DIR, "*.txt"))

    # Line 19-20: Isolate the raw names of the files without extensions.
    image_basenames = {os.path.splitext(os.path.basename(f))[0] for f in image_files}
    label_basenames = {os.path.splitext(os.path.basename(f))[0] for f in label_files}

    print("--- Dataset Overview ---")
    print(f"Total Images Found: {len(image_files)}")
    print(f"Total Label Files Found: {len(label_files)}")

    # Line 26-27: Compare the sets to see if any pair is broken or missing.
    missing_labels = image_basenames - label_basenames
    missing_images = label_basenames - image_basenames

    print("\n--- Integrity Checks ---")
    if not missing_labels and not missing_images:
        print("✓ Perfect Score: Every picture has a label file!")
    else:
        if missing_labels:
            print(f"✗ Error: These images don't have labels: {missing_labels}")
        if missing_images:
            print(f"✗ Error: These labels don't have images: {missing_images}")

    # Line 38-40: Initialize counters to track box counts.
    class_counter = Counter()
    total_annotations = 0
    empty_labels = 0

    # Line 42: Look into every label text file item by item.
    for label_path in label_files:
        if os.path.getsize(label_path) == 0:
            empty_labels += 1
            continue

        # Line 47: Read inside the file to count the bounding boxes drawn.
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_id = int(parts[0])
                    class_name = CLASS_MAP.get(class_id, f"Unknown ({class_id})")
                    class_counter[class_name] += 1
                    total_annotations += 1

    print("\n--- Annotation Breakdown ---")
    print(f"Total Bounding Boxes Drawn: {total_annotations}")
    print(f"Images with no boxes inside: {empty_labels}")
    print("\nTotal boxes per class:")
    for cls_name, count in class_counter.items():
        print(f"  - {cls_name}: {count}")

# Line 61: Boot up our inspection function!
validate_dataset()

--- Dataset Overview ---
Total Images Found: 0
Total Label Files Found: 0

--- Integrity Checks ---
✓ Perfect Score: Every picture has a label file!

--- Annotation Breakdown ---
Total Bounding Boxes Drawn: 0
Images with no boxes inside: 0

Total boxes per class:
